<a href="https://colab.research.google.com/github/cto-school/mathematics-for-machine-learning/blob/main/01-numpy-for-mathematics/01b_vectorization_broadcasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 01b — Vectorization & Broadcasting

This is where NumPy earns its keep. The theme of this whole notebook:

> **Turn an equation directly into array code.**

If a formula says "do this to every entry", you do **not** write a loop — you
write the formula once and NumPy applies it to the entire array at once. This is
called **vectorization**. It is shorter, reads like math, and runs *much* faster.

## 1. Elementwise arithmetic

Arithmetic on arrays happens **elementwise**: `u + v` adds matching entries,
`u * v` multiplies matching entries, and so on. This is exactly vector addition
and scalar multiplication from linear algebra.

In [ ]:
import numpy as np

u = np.array([1, 2, 3])
v = np.array([10, 20, 30])

print(u + v)     # [11 22 33]   entry by entry
print(v - u)     # [ 9 18 27]
print(u * v)     # [10 40 90]   elementwise product (NOT the dot product)
print(v / u)     # [10. 10. 10.]
print(u ** 2)    # [1 4 9]      square every entry

In [ ]:
# A scalar applies to every entry (this is "broadcasting", section 4):
print(3 * u)     # [3 6 9]   scalar multiple of a vector
print(u + 100)   # [101 102 103]

## 2. Universal functions (ufuncs): math on whole arrays

NumPy ships fast, elementwise versions of the usual math functions. They are
called **ufuncs**. Feed in an array, get back an array of the same shape.

$\sin,\ \cos,\ \exp,\ \log,\ \sqrt,\ \dots$ all work this way.

In [ ]:
import numpy as np

x = np.array([0.0, 1.0, 4.0, 9.0])

print(np.sqrt(x))   # [0. 1. 2. 3.]    square root of each entry
print(np.exp(x))    # e^x for each entry
print(np.log1p(x))  # log(1+x), safe near 0

In [ ]:
# Evaluate sin at a few special angles, all at once:
angles = np.array([0, np.pi / 6, np.pi / 4, np.pi / 2])
print(np.sin(angles))    # [0.   0.5  0.707 1. ]  (approx)
print(np.cos(angles))

**Turning an equation into code.** Suppose $g(x) = \sqrt{x}\,e^{-x}$. On paper
that's one line; in NumPy it is also one line, and it works on a whole array of
$x$ values simultaneously:

In [ ]:
import numpy as np

x = np.linspace(0, 4, 9)         # nine x-values
g = np.sqrt(x) * np.exp(-x)      # the formula, applied to ALL of them at once
for xi, gi in zip(x, g):
    print(f"g({xi:.1f}) = {gi:.4f}")

## 3. Vectorization vs. Python loops (and why speed matters)

The same computation can be done with a slow Python loop or with one vectorized
expression. Let's check they agree, then time them.

In [ ]:
import numpy as np

N = 1_000_000
x = np.linspace(0, 10, N)

# --- Way 1: a plain Python loop (slow) ---
y_loop = np.empty(N)
for i in range(N):
    y_loop[i] = np.sin(x[i]) ** 2     # work on one element at a time

# --- Way 2: vectorized (fast) ---
y_vec = np.sin(x) ** 2                 # one expression, whole array

# Do they give the same numbers? (allclose checks "equal up to tiny rounding")
print("same result?", np.allclose(y_loop, y_vec))

In [ ]:
import numpy as np
import time

N = 1_000_000
x = np.linspace(0, 10, N)

# Time the loop
t0 = time.perf_counter()
y_loop = np.empty(N)
for i in range(N):
    y_loop[i] = np.sin(x[i]) ** 2
t1 = time.perf_counter()

# Time the vectorized version
t2 = time.perf_counter()
y_vec = np.sin(x) ** 2
t3 = time.perf_counter()

loop_time = t1 - t0
vec_time = t3 - t2
print(f"loop       : {loop_time:.4f} s")
print(f"vectorized : {vec_time:.4f} s")
print(f"speedup    : about {loop_time / vec_time:.0f}x faster")

The vectorized version is typically **tens to hundreds of times faster** — and
shorter to write. The lesson: *if you find yourself writing a `for` loop over the
entries of an array, look for the vectorized expression instead.*

## 4. Broadcasting: combining different shapes

What if shapes don't match? NumPy **broadcasts**: it stretches the smaller array
across the larger one when their shapes are compatible. You already saw the
simplest case, `3 * u`, where the scalar `3` was applied to every entry.

**The rule (read shapes from the right):** two dimensions are compatible when
they are **equal**, or **one of them is 1**. A size-1 (or missing) dimension is
stretched to match.

In [ ]:
import numpy as np

A = np.array([[1, 2, 3],
              [4, 5, 6]])      # shape (2, 3)
row = np.array([10, 20, 30])   # shape (3,)

# 'row' is stretched down to both rows of A:
print(A + row)
# [[11 22 33]
#  [14 25 36]]

In [ ]:
import numpy as np

A = np.array([[1, 2, 3],
              [4, 5, 6]])          # shape (2, 3)
col = np.array([[100],
                [200]])            # shape (2, 1)

# 'col' is stretched across the three columns:
print(A + col)
# [[101 102 103]
#  [204 205 206]]

A classic use: build a **multiplication table** (an outer product) by combining a
column vector and a row vector. The $(i,j)$ entry becomes $i \cdot j$.

In [ ]:
import numpy as np

i = np.arange(1, 6).reshape(5, 1)   # column 1..5, shape (5, 1)
j = np.arange(1, 6).reshape(1, 5)   # row    1..5, shape (1, 5)

table = i * j                        # broadcasting -> 5x5 product table
print(table)

## 5. Reductions: collapsing an array to a summary

A **reduction** boils an array down to fewer numbers: `sum`, `mean`, `std`
(standard deviation), `min`, `max`. With no `axis`, it uses the whole array.

In [ ]:
import numpy as np

x = np.array([2.0, 4.0, 4.0, 4.0, 5.0, 5.0, 7.0, 9.0])
print("sum  :", x.sum())
print("mean :", x.mean())
print("std  :", x.std())     # population standard deviation
print("min  :", x.min(), " max :", x.max())

For a 2D array, `axis` chooses the direction to collapse:

- `axis=0` collapses **down the rows** → one value per **column**.
- `axis=1` collapses **across the columns** → one value per **row**.

In [ ]:
import numpy as np

A = np.array([[1, 2, 3],
              [4, 5, 6]])      # shape (2, 3)

print("total          :", A.sum())          # 21  (everything)
print("column sums    :", A.sum(axis=0))    # [5 7 9]   (down the rows)
print("row sums       :", A.sum(axis=1))    # [ 6 15]   (across the columns)
print("column means   :", A.mean(axis=0))   # [2.5 3.5 4.5]

## 6. The payoff: an equation becomes a plot

Now we combine everything. To plot $f(x) = \sin(x)$ on $[0, 2\pi]$ we make a fine
grid with `linspace`, apply the ufunc `np.sin` to the whole grid, and plot. No
loops anywhere — the code reads like the math.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(0, 2 * np.pi, 200)   # 200 points across [0, 2*pi]
y = np.sin(x)                        # apply sin to ALL of them at once

plt.plot(x, y)
plt.title(r"$y = \sin(x)$")
plt.xlabel("x")
plt.ylabel("sin(x)")
plt.axhline(0, color="gray", linewidth=0.8)   # the x-axis
plt.grid(True)
plt.show()

Compare this to the list-and-loop version from Chapter 00b. Same picture, far
less code — and it generalizes: swap `np.sin` for any ufunc and you instantly
plot a different function.

---
## ✍️ Exercise 1 (solution included)

Without any loop, evaluate the **Gaussian / bell curve**

$$f(x) = e^{-x^2}$$

on 201 points over $[-3, 3]$, and report its maximum value and where (which
$x$) the maximum occurs. *Hint:* `np.argmax` gives the **index** of the largest
entry.

**Solution:**

In [ ]:
import numpy as np

x = np.linspace(-3, 3, 201)
f = np.exp(-x ** 2)            # the formula, vectorized

print("max value :", f.max())             # 1.0
i = np.argmax(f)                           # index of the largest entry
print("at x      :", x[i])                 # 0.0  (the peak sits at x=0)

## ✍️ Exercise 2 (solution included)

You are given a small data matrix where each **row** is a sample and each
**column** is a feature. **Standardize** each column so it has mean 0 and
standard deviation 1, i.e. replace each entry by

$$z = \frac{x - \mu_{\text{col}}}{\sigma_{\text{col}}}.$$

Do it in one broadcasted expression (no loops). Then verify the new column means
are ~0 and column standard deviations are ~1.

**Solution:**

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
X = rng.normal(loc=5.0, scale=2.0, size=(6, 3))   # 6 samples, 3 features

mu = X.mean(axis=0)     # one mean per column,  shape (3,)
sd = X.std(axis=0)      # one std  per column,  shape (3,)

Z = (X - mu) / sd       # broadcasting subtracts/divides per column

print("new column means:", np.round(Z.mean(axis=0), 10))  # ~ [0 0 0]
print("new column stds :", np.round(Z.std(axis=0), 10))   # ~ [1 1 1]

---
## 📝 Homework (no solutions provided)

1. On a grid `x = np.linspace(0, 2*np.pi, 300)`, plot $\sin(x)$ and $\cos(x)$ on
   the **same** axes. (Call `plt.plot` twice before `plt.show()`.)
2. Verify the identity $\sin^2 x + \cos^2 x = 1$ numerically: compute
   `np.sin(x)**2 + np.cos(x)**2` on a grid and check it is `1` everywhere with
   `np.allclose`.
3. Build the $10\times 10$ multiplication table using broadcasting, then use a
   reduction to find the sum of all its entries and the largest value in each
   row.
4. Implement the **sigmoid** $\sigma(x) = \dfrac{1}{1 + e^{-x}}$ as a one-line
   vectorized function, evaluate it on `np.linspace(-6, 6, 200)`, and plot it.
   Where does the curve cross $y = 0.5$?